In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jaypatel1054","key":"cc2f4db1d00aa547e2db881cf9daa548"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d suraj520/customer-support-ticket-dataset
!unzip -q customer-support-ticket-dataset.zip -d ticket_data

Dataset URL: https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset
License(s): CC0-1.0
100% 828k/828k [00:00<00:00, 98.0MB/s]



In [ ]:
import pandas as pd

df = pd.read_csv("ticket_data/customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
print(df.shape)
print(df['Ticket Type'].value_counts())
print(df.isnull().sum())

(8469, 17)
Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64
Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64


In [ ]:
department_map = {
    "Technical issue": "Technical",
    "Billing inquiry": "Billing",
    "Refund request": "Billing",
    "Cancellation request": "Account",
    "Product inquiry": "General"
}

df["department"] = df["Ticket Type"].map(department_map)

df["department"].value_counts()

,count
department,
Billing,3386
Technical,1747
Account,1695
General,1641


In [ ]:
high_keywords = ["urgent", "immediately", "asap", "not working", "down",
                  "can't access", "critical", "emergency", "charged twice",
                  "unauthorized", "locked out", "failed"]

medium_keywords = ["issue", "problem", "error", "not sure", "help", "trouble"]

def assign_urgency(text):
    text = str(text).lower()
    if any(keyword in text for keyword in high_keywords):
        return "High"
    elif any(keyword in text for keyword in medium_keywords):
        return "Medium"
    else:
        return "Low"

In [ ]:
df["combined_text"] = df["Ticket Subject"] + " " + df["Ticket Description"]

df["urgency"] = df["combined_text"].apply(assign_urgency)

df["urgency"].value_counts()

,count
urgency,
Medium,7854
High,530
Low,85


In [ ]:
print(df["combined_text"].str.contains("issue", case=False).sum())
print(df["combined_text"].str.contains("problem", case=False).sum())

7847
3186


In [ ]:
medium_keywords = ["not sure", "trouble", "error", "confused", "delay", "slow", "help me"]

for kw in high_keywords + medium_keywords:
    count = df["combined_text"].str.contains(kw, case=False, regex=False).sum()
    print(f"{kw!r}: {count} tickets ({count/len(df)*100:.1f}%)")

'urgent': 166 tickets (2.0%)
'immediately': 31 tickets (0.4%)
'asap': 14 tickets (0.2%)
'not working': 210 tickets (2.5%)
'down': 108 tickets (1.3%)
"can't access": 1 tickets (0.0%)
'critical': 2 tickets (0.0%)
'emergency': 2 tickets (0.0%)
'charged twice': 0 tickets (0.0%)
'unauthorized': 0 tickets (0.0%)
'locked out': 0 tickets (0.0%)
'failed': 9 tickets (0.1%)
'not sure': 283 tickets (3.3%)
'trouble': 1428 tickets (16.9%)
'error': 987 tickets (11.7%)
'confused': 5 tickets (0.1%)
'delay': 5 tickets (0.1%)
'slow': 1 tickets (0.0%)
'help me': 247 tickets (2.9%)


In [ ]:
df["urgency"] = df["combined_text"].apply(assign_urgency)
df["urgency"].value_counts()

,count
urgency,
Low,5342
Medium,2597
High,530


In [ ]:
pd.set_option('display.max_colwidth', 150)
df[df["urgency"] == "High"][["combined_text", "urgency"]].sample(5)

,combined_text,urgency
1028,"Hardware issue I've forgotten my password for my {product_purchased} account, and the password reset option is not working. How can I recover my a...",High
3922,"Payment issue I've forgotten my password for my {product_purchased} account, and the password reset option is not working. How can I recover my ac...",High
1563,Refund request I've accidentally deleted important data from my {product_purchased}. Is there any way to recover the deleted files? I need them ur...,High
5816,"Product setup I'm having an issue with the {product_purchased}. Please assist.\n\nWe don't care what device you use. When you take photos, you wan...",High
7730,Peripheral compatibility I'm facing issues logging into my {product_purchased} account. It says my account is locked. What should I do to unlock i...,High


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = "I need a refund immediately, my card was charged twice!"
tokens = tokenizer(sample_text)
print(tokens)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 1045, 2342, 1037, 25416, 8630, 3202, 1010, 2026, 4003, 2001, 5338, 3807, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
tokens_readable = tokenizer.convert_ids_to_tokens(tokens["input_ids"])
print(tokens_readable)

['[CLS]', 'i', 'need', 'a', 'ref', '##und', 'immediately', ',', 'my', 'card', 'was', 'charged', 'twice', '!', '[SEP]']


In [ ]:
lengths = df["combined_text"].apply(lambda x: len(tokenizer.tokenize(x)))
print(lengths.describe())

count    8469.000000
mean       71.267210
std        10.159453
min        35.000000
25%        68.000000
50%        73.000000
75%        77.000000
max       201.000000
Name: combined_text, dtype: float64


In [ ]:
encodings = tokenizer(
    list(df["combined_text"]),
    padding="max_length",
    truncation=True,
    max_length=128
)

print(len(encodings["input_ids"]))
print(encodings["input_ids"][0])

8469
[101, 4031, 16437, 1045, 1005, 1049, 2383, 2019, 3277, 2007, 1996, 1063, 4031, 1035, 4156, 1065, 1012, 3531, 6509, 1012, 2115, 25640, 14101, 3642, 2003, 1024, 6390, 19841, 2487, 1012, 2057, 9120, 2008, 2017, 2031, 7303, 1037, 4037, 4769, 1012, 3531, 3313, 4638, 2115, 10373, 4769, 1012, 1045, 1005, 2310, 2699, 13460, 23416, 2075, 4084, 3855, 1999, 1996, 5310, 6410, 1010, 2021, 1996, 3277, 29486, 2015, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
print(len(encodings["input_ids"][0]))
print(encodings["input_ids"][0].count(0))

128
60


In [ ]:
print(encodings["attention_mask"][0])
print(encodings["attention_mask"][0].count(1))
print(encodings["attention_mask"][0].count(0))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
68
60


In [ ]:
department_labels = sorted(df["department"].unique())
dept2id = {label: idx for idx, label in enumerate(department_labels)}
id2dept = {idx: label for label, idx in dept2id.items()}

df["department_id"] = df["department"].map(dept2id)

print(dept2id)
print(df["department_id"].value_counts())

{'Account': 0, 'Billing': 1, 'General': 2, 'Technical': 3}
department_id
1    3386
3    1747
0    1695
2    1641
Name: count, dtype: int64


In [ ]:
urgency_labels = sorted(df["urgency"].unique())
urg2id = {label: idx for idx, label in enumerate(urgency_labels)}
id2urg = {idx: label for label, idx in urg2id.items()}

df["urgency_id"] = df["urgency"].map(urg2id)

print(urg2id)
print(df["urgency_id"].value_counts())

{'High': 0, 'Low': 1, 'Medium': 2}
urgency_id
1    5342
2    2597
0     530
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["department_id"]
)

print(train_df.shape)
print(test_df.shape)
print(train_df["department_id"].value_counts(normalize=True))
print(test_df["department_id"].value_counts(normalize=True))

(6775, 22)
(1694, 22)
department_id
1    0.399852
3    0.206199
0    0.200148
2    0.193801
Name: proportion, dtype: float64
department_id
1    0.399646
3    0.206612
0    0.200118
2    0.193625
Name: proportion, dtype: float64


In [ ]:
print(train_df["urgency_id"].value_counts(normalize=True))
print(test_df["urgency_id"].value_counts(normalize=True))

urgency_id
1    0.628487
2    0.309668
0    0.061845
Name: proportion, dtype: float64
urgency_id
1    0.639906
2    0.294569
0    0.065525
Name: proportion, dtype: float64


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model.config.num_labels)
print(model.classifier)

4
Linear(in_features=768, out_features=4, bias=True)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_department",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [ ]:
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(examples["combined_text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[["combined_text", "department_id"]])
test_dataset = Dataset.from_pandas(test_df[["combined_text", "department_id"]])

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("department_id", "labels")
test_dataset = test_dataset.rename_column("department_id", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/6775 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['combined_text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6775
})
{'labels': tensor(1), 'input_ids': tensor([  101,  4031, 16437,  1045,  1005,  1049,  2383,  2019,  3277,  2007,
         1996,  1063,  4031,  1035,  4156,  1065,  1012,  3531,  6509,  1012,
         4067,  2017,  1012,  1000,  1045,  2031,  1037,  1031,  8136,  3457,
         1033,  2007,  2033,  2043,  1996,  3124,  3310,  1999,  1998,  2026,
         3611,  2758,  1005,  1045,  2215,  2000,  2393,  1012,  1045,  2215,
         1045,  1005,  2310,  2864,  1037,  4713, 25141,  2006,  2026,  1063,
         4031,  1035,  4156,  1065,  1010,  5327,  2009,  2052, 10663,  1996,
         3291,  1010,  2021,  2009,  2134,  1005,  1056,  2393,  1012,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.345057,1.335093,0.399646,0.228225,0.159717,0.399646
2,1.340658,1.337480,0.399646,0.228225,0.159717,0.399646
3,1.331421,1.345209,0.399646,0.228225,0.159717,0.399646


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1272, training_loss=1.3322795711973179, metrics={'train_runtime': 258.2402, 'train_samples_per_second': 78.706, 'train_steps_per_second': 4.926, 'total_flos': 673123976985600.0, 'train_loss': 1.3322795711973179, 'epoch': 3.0})

In [ ]:
import numpy as np
from collections import Counter

predictions_output = trainer.predict(test_dataset)
preds = np.argmax(predictions_output.predictions, axis=1)
print(Counter(preds.tolist()))

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Counter({1: 1694})


In [ ]:
duplicate_check = df.groupby("Ticket Description")["department"].nunique().sort_values(ascending=False)
print(duplicate_check.head(10))
print((duplicate_check > 1).sum(), "out of", len(duplicate_check), "unique descriptions map to more than one department")

Ticket Description
I'm having an issue with the {product_purchased}. Please assist. I'm unable to find the option to perform the desired action in the {product_purchased}. Could you please guide me through the steps?           4
I'm having an issue with the {product_purchased}. Please assist. I'm not sure if this issue is specific to my device or if others have reported similar problems.                                               4
I'm having an issue with the {product_purchased}. Please assist. I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?    4
I'm having an issue with the {product_purchased}. Please assist. This problem started occurring after the recent software update. I haven't made any other changes to the device.                               4
I'm having an issue with the {product_purchased}. Please assist. I've tried troubleshooting steps mentioned in the user manual, but the issue

In [ ]:
pd.crosstab(df["Ticket Subject"], df["department"]).sample(10, random_state=1)

department,Account,Billing,General,Technical
Ticket Subject,,,,
Data loss,115,186,91,99
Product setup,108,208,100,113
Installation support,99,227,99,105
Cancellation request,103,191,85,108
Hardware issue,109,229,106,103
Peripheral compatibility,102,200,93,101
Delivery problem,114,222,109,116
Battery life,104,225,101,112
Refund request,126,219,122,109


In [ ]:
department_keywords = {
    "invoice": None, "charge": None, "charged": None, "billing": None,
    "payment": None, "refund": None, "credit card": None, "subscription": None,
    "not working": None, "error": None, "bug": None, "crash": None,
    "device": None, "software": None, "hardware": None, "firmware": None, "network": None,
    "password": None, "login": None, "log in": None, "locked out": None,
    "cancel": None, "cancellation": None, "username": None, "sign in": None
}

for kw in department_keywords:
    count = df["combined_text"].str.contains(kw, case=False, regex=False).sum()
    print(f"{kw!r}: {count} tickets ({count/len(df)*100:.1f}%)")

'invoice': 38 tickets (0.4%)
'charge': 325 tickets (3.8%)
'charged': 35 tickets (0.4%)
'billing': 23 tickets (0.3%)
'payment': 610 tickets (7.2%)
'refund': 723 tickets (8.5%)
'credit card': 58 tickets (0.7%)
'subscription': 13 tickets (0.2%)
'not working': 210 tickets (2.5%)
'error': 987 tickets (11.7%)
'bug': 964 tickets (11.4%)
'crash': 338 tickets (4.0%)
'device': 1502 tickets (17.7%)
'software': 1935 tickets (22.8%)
'hardware': 1118 tickets (13.2%)
'firmware': 495 tickets (5.8%)
'network': 1056 tickets (12.5%)
'password': 230 tickets (2.7%)
'login': 243 tickets (2.9%)
'log in': 51 tickets (0.6%)
'locked out': 0 tickets (0.0%)
'cancel': 523 tickets (6.2%)
'cancellation': 488 tickets (5.8%)
'username': 17 tickets (0.2%)
'sign in': 2 tickets (0.0%)


In [ ]:
pd.crosstab(df["combined_text"].str.contains("error", case=False), df["department"])

department,Account,Billing,General,Technical
combined_text,,,,
False,1497,2982,1461,1542
True,198,404,180,205


In [ ]:
billing_keywords = ["invoice", "charge", "charged", "billing", "payment",
                     "refund", "credit card", "subscription"]

technical_keywords = ["not working", "device", "software", "hardware",
                       "firmware", "network", "crash"]

account_keywords = ["password", "login", "log in", "locked out",
                     "cancel", "cancellation", "username", "sign in"]

def assign_department(text):
    text = str(text).lower()
    if any(keyword in text for keyword in billing_keywords):
        return "Billing"
    elif any(keyword in text for keyword in technical_keywords):
        return "Technical"
    elif any(keyword in text for keyword in account_keywords):
        return "Account"
    else:
        return "General"

In [ ]:
df["department_v2"] = df["combined_text"].apply(assign_department)
df["department_v2"].value_counts()

,count
department_v2,
Technical,3973
General,2459
Billing,1675
Account,362


In [ ]:
pd.crosstab(df["Ticket Subject"], df["department_v2"]).sample(10, random_state=1)

department_v2,Account,Billing,General,Technical
Ticket Subject,,,,
Data loss,11,37,221,222
Product setup,14,45,259,211
Installation support,16,44,246,224
Cancellation request,224,44,0,219
Hardware issue,0,36,0,511
Peripheral compatibility,17,40,240,199
Delivery problem,18,42,264,237
Battery life,8,43,265,226
Refund request,0,576,0,0


In [ ]:
department_labels_v2 = sorted(df["department_v2"].unique())
dept2id_v2 = {label: idx for idx, label in enumerate(department_labels_v2)}
id2dept_v2 = {idx: label for label, idx in dept2id_v2.items()}

df["department_id_v2"] = df["department_v2"].map(dept2id_v2)

print(dept2id_v2)
print(df["department_id_v2"].value_counts())

{'Account': 0, 'Billing': 1, 'General': 2, 'Technical': 3}
department_id_v2
3    3973
2    2459
1    1675
0     362
Name: count, dtype: int64


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["department_id_v2"]
)

print(train_df.shape)
print(test_df.shape)
print(train_df["department_id_v2"].value_counts(normalize=True))
print(test_df["department_id_v2"].value_counts(normalize=True))

(6775, 24)
(1694, 24)
department_id_v2
3    0.469077
2    0.290332
1    0.197786
0    0.042804
Name: proportion, dtype: float64
department_id_v2
3    0.469303
2    0.290437
1    0.197757
0    0.042503
Name: proportion, dtype: float64


In [ ]:
train_dataset = Dataset.from_pandas(train_df[["combined_text", "department_id_v2"]])
test_dataset = Dataset.from_pandas(test_df[["combined_text", "department_id_v2"]])

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("department_id_v2", "labels")
test_dataset = test_dataset.rename_column("department_id_v2", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_dataset[0])

Map:   0%|          | 0/6775 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

{'labels': tensor(3), 'input_ids': tensor([  101,  8051,  3277,  1045,  1005,  1049,  2383,  2019,  3277,  2007,
         1996,  1063,  4031,  1035,  4156,  1065,  1012,  3531,  6509,  1012,
         2017,  2097,  2022,  2583,  2000,  7276,  1996,  4031,  2000, 16500,
         1012,  2023,  2071,  3828,  2017,  2116,  5190,  2065,  1996,  4031,
         2003,  2525,  1999,  1037,  2291,  2030,  2089,  2022,  2583,  2000,
         2022,  5361,  2302,  1996,  3277,  1045,  1005,  1049,  5307,  2003,
        23852,  1012,  2823,  2009,  2573,  2986,  1010,  2021,  2060,  2335,
         2009,  4490,  2039, 14153,  1012,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.031948,0.031179,0.994687,0.994669,0.994761,0.994687
2,0.014066,0.008855,0.997639,0.997634,0.997658,0.997639
3,0.008627,0.005464,0.998229,0.998229,0.998240,0.998229


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1272, training_loss=0.06214479342653598, metrics={'train_runtime': 264.806, 'train_samples_per_second': 76.754, 'train_steps_per_second': 4.804, 'total_flos': 673123976985600.0, 'train_loss': 0.06214479342653598, 'epoch': 3.0})

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

predictions_output = trainer.predict(test_dataset)
preds = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

print(classification_report(true_labels, preds, target_names=department_labels_v2))

              precision    recall  f1-score   support

     Account       0.99      1.00      0.99        72
     Billing       1.00      1.00      1.00       335
     General       1.00      0.99      1.00       492
   Technical       1.00      1.00      1.00       795

    accuracy                           1.00      1694
   macro avg       1.00      1.00      1.00      1694
weighted avg       1.00      1.00      1.00      1694



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
X_train = vectorizer.fit_transform(train_df["combined_text"])
X_test = vectorizer.transform(test_df["combined_text"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df["department_id_v2"])

base_preds = clf.predict(X_test)
print(classification_report(test_df["department_id_v2"], base_preds, target_names=department_labels_v2))

              precision    recall  f1-score   support

     Account       0.98      0.71      0.82        72
     Billing       1.00      0.81      0.89       335
     General       0.87      1.00      0.93       492
   Technical       0.96      0.98      0.97       795

    accuracy                           0.94      1694
   macro avg       0.95      0.87      0.90      1694
weighted avg       0.94      0.94      0.94      1694



In [ ]:
robust_tickets = [
    # Billing (money)
    ("Money got taken out of my bank two times for the same order.", "Billing"),
    ("The amount on my monthly bill is higher than what I was quoted.", "Billing"),
    ("I returned the item three weeks ago and still haven't got my money back.", "Billing"),
    ("Why was I taxed extra on my last order? The total looks wrong.", "Billing"),
    ("My card was declined at checkout even though I have enough funds.", "Billing"),
    ("I need a copy of the receipt for my purchase last month.", "Billing"),
    ("You took my money for the yearly plan but I only wanted the monthly one.", "Billing"),
    ("There is an unexpected fee on my statement that I don't recognise.", "Billing"),
    ("Can I get a discount code applied after I already paid?", "Billing"),
    ("I was promised a coupon credit but it never showed up on my balance.", "Billing"),
    ("The price at checkout was different from the price on the website.", "Billing"),
    ("I paid twice for the same order and only received one item.", "Billing"),
    # Technical (something broken)
    ("My laptop screen goes black after a few minutes and I have to restart it.", "Technical"),
    ("The app freezes every time I open the settings page.", "Technical"),
    ("Wi-Fi keeps dropping whenever I stream videos.", "Technical"),
    ("My phone gets very hot and the battery drains within an hour.", "Technical"),
    ("The printer will not connect to my computer anymore.", "Technical"),
    ("Sound cuts out randomly while I'm on video calls.", "Technical"),
    ("The camera shows a blank screen when I start the program.", "Technical"),
    ("My smart TV won't turn on and the red light keeps blinking.", "Technical"),
    ("The update froze halfway and now the machine won't start.", "Technical"),
    ("Files take forever to load and the whole system is extremely slow.", "Technical"),
    ("The keyboard types the wrong letters after I plugged it in.", "Technical"),
    ("The website shows a blank white page on every browser I try.", "Technical"),
    # Account (access and account management)
    ("I can't get into my profile since yesterday and the reset email never arrives.", "Account"),
    ("Please close my membership at the end of this month.", "Account"),
    ("I want to change the email address linked to my account.", "Account"),
    ("Someone else may have got into my profile, I see activity I didn't do.", "Account"),
    ("How do I delete my account and all my data permanently?", "Account"),
    ("I forgot the phrase I use to access my profile and can't recover it.", "Account"),
    ("Please end my plan and stop the automatic renewal.", "Account"),
    ("I want to upgrade my membership to the premium tier.", "Account"),
    ("The verification code never reaches my phone when I try to access my profile.", "Account"),
    ("My account name is spelled wrong and I can't edit it.", "Account"),
    ("Please merge my two profiles into a single one.", "Account"),
    ("I want to add my colleague as a second user on my membership.", "Account"),
    # General (questions before or around buying)
    ("Do you ship to Canada and how long does delivery usually take?", "General"),
    ("What are your opening hours on weekends?", "General"),
    ("Is this model compatible with the older version of your accessory?", "General"),
    ("Which colors does the new model come in?", "General"),
    ("Do you offer a student discount?", "General"),
    ("Can you tell me how long the warranty lasts?", "General"),
    ("I'd like to know the differences between the two premium models.", "General"),
    ("Where is the nearest store where I can try the product before buying?", "General"),
    ("Are there any upcoming sales events this month?", "General"),
    ("Do you have a user manual available in Spanish?", "General"),
    ("How do I contact the sales team for a bulk order?", "General"),
    ("Is the product waterproof?", "General"),
]

robust_df = pd.DataFrame(robust_tickets, columns=["combined_text", "human_label"])
print(robust_df["human_label"].value_counts())

all_keywords = billing_keywords + technical_keywords + account_keywords

def find_leaks(text):
    text = text.lower()
    return [kw for kw in all_keywords if kw in text]

robust_df["leaked_keywords"] = robust_df["combined_text"].apply(find_leaks)
print(robust_df[robust_df["leaked_keywords"].str.len() > 0])

human_label
Billing      12
Technical    12
Account      12
General      12
Name: count, dtype: int64
Empty DataFrame
Columns: [combined_text, human_label, leaked_keywords]
Index: []


In [ ]:
robust_df["labels"] = robust_df["human_label"].map(dept2id_v2)
print(robust_df["labels"].isnull().sum())

robust_dataset = Dataset.from_pandas(robust_df[["combined_text", "labels"]])
robust_dataset = robust_dataset.map(tokenize_function, batched=True)
robust_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

bert_out = trainer.predict(robust_dataset)
bert_preds = np.argmax(bert_out.predictions, axis=1)

X_rob = vectorizer.transform(robust_df["combined_text"])
base_preds_rob = clf.predict(X_rob)

print("DistilBERT")
print(classification_report(robust_df["labels"], bert_preds, target_names=department_labels_v2, zero_division=0))
print("Baseline")
print(classification_report(robust_df["labels"], base_preds_rob, target_names=department_labels_v2, zero_division=0))

0


Map:   0%|          | 0/48 [00:00<?, ? examples/s]

DistilBERT
              precision    recall  f1-score   support

     Account       0.50      0.17      0.25        12
     Billing       1.00      0.25      0.40        12
     General       0.30      0.92      0.45        12
   Technical       1.00      0.33      0.50        12

    accuracy                           0.42        48
   macro avg       0.70      0.42      0.40        48
weighted avg       0.70      0.42      0.40        48

Baseline
              precision    recall  f1-score   support

     Account       0.00      0.00      0.00        12
     Billing       1.00      0.33      0.50        12
     General       0.29      0.17      0.21        12
   Technical       0.30      0.92      0.45        12

    accuracy                           0.35        48
   macro avg       0.40      0.35      0.29        48
weighted avg       0.40      0.35      0.29        48



In [ ]:
review_df = robust_df.copy()
review_df["bert_pred"] = [id2dept_v2[p] for p in bert_preds]
review_df["baseline_pred"] = [id2dept_v2[p] for p in base_preds_rob]

wrong_bert = review_df[review_df["human_label"] != review_df["bert_pred"]]
pd.set_option('display.max_colwidth', 100)
print(wrong_bert[["combined_text", "human_label", "bert_pred"]].to_string())

                                                                     combined_text human_label bert_pred
0                     Money got taken out of my bank two times for the same order.     Billing   General
2         I returned the item three weeks ago and still haven't got my money back.     Billing   General
4                My card was declined at checkout even though I have enough funds.     Billing   Account
5                         I need a copy of the receipt for my purchase last month.     Billing   General
6         You took my money for the yearly plan but I only wanted the monthly one.     Billing   General
7               There is an unexpected fee on my statement that I don't recognise.     Billing   General
9             I was promised a coupon credit but it never showed up on my balance.     Billing   General
10              The price at checkout was different from the price on the website.     Billing   General
11                     I paid twice for the same order 

In [ ]:
wrong_base = review_df[review_df["human_label"] != review_df["baseline_pred"]]
print(wrong_base[["combined_text", "human_label", "baseline_pred"]].to_string())

                                                                     combined_text human_label baseline_pred
1                  The amount on my monthly bill is higher than what I was quoted.     Billing       General
2         I returned the item three weeks ago and still haven't got my money back.     Billing     Technical
3                   Why was I taxed extra on my last order? The total looks wrong.     Billing       General
5                         I need a copy of the receipt for my purchase last month.     Billing     Technical
6         You took my money for the yearly plan but I only wanted the monthly one.     Billing       General
7               There is an unexpected fee on my statement that I don't recognise.     Billing     Technical
8                          Can I get a discount code applied after I already paid?     Billing     Technical
10              The price at checkout was different from the price on the website.     Billing       General
17                 

In [ ]:
train_dataset_urg = Dataset.from_pandas(train_df[["combined_text", "urgency_id"]])
test_dataset_urg = Dataset.from_pandas(test_df[["combined_text", "urgency_id"]])

train_dataset_urg = train_dataset_urg.map(tokenize_function, batched=True)
test_dataset_urg = test_dataset_urg.map(tokenize_function, batched=True)

train_dataset_urg = train_dataset_urg.rename_column("urgency_id", "labels")
test_dataset_urg = test_dataset_urg.rename_column("urgency_id", "labels")

train_dataset_urg.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset_urg.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

urgency_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

training_args_urg = TrainingArguments(
    output_dir="./results_urgency",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

trainer_urg = Trainer(
    model=urgency_model,
    args=training_args_urg,
    train_dataset=train_dataset_urg,
    eval_dataset=test_dataset_urg,
    compute_metrics=compute_metrics
)

trainer_urg.train()

Map:   0%|          | 0/6775 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.142820,0.103093,0.979339,0.978761,0.979760,0.979339
2,0.060130,0.064447,0.989374,0.988948,0.989487,0.989374
3,0.043819,0.054093,0.989965,0.989696,0.989809,0.989965


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1272, training_loss=0.11488749016568346, metrics={'train_runtime': 262.5755, 'train_samples_per_second': 77.406, 'train_steps_per_second': 4.844, 'total_flos': 673111973203200.0, 'train_loss': 0.11488749016568346, 'epoch': 3.0})

In [ ]:
urg_out = trainer_urg.predict(test_dataset_urg)
urg_preds = np.argmax(urg_out.predictions, axis=1)
print(classification_report(test_df["urgency_id"], urg_preds, target_names=urgency_labels, zero_division=0))

              precision    recall  f1-score   support

        High       0.97      0.86      0.91       102
         Low       0.99      1.00      0.99      1078
      Medium       0.99      1.00      0.99       514

    accuracy                           0.99      1694
   macro avg       0.98      0.95      0.97      1694
weighted avg       0.99      0.99      0.99      1694



In [ ]:
urgency_robust = [
    ("My whole website has been offline since this morning and customers can't check out.", "High"),
    ("I think someone else is using my card right now, there are charges I didn't make.", "High"),
    ("All my files just vanished and I have a client meeting in an hour.", "High"),
    ("My account got suspended and I can't reach my team's shared documents at all.", "High"),
    ("The server keeps crashing every few minutes and we're losing customers.", "High"),
    ("I'm locked out and my card on file already got billed for this month.", "High"),
    ("Whenever you get a chance, could you look into a small display glitch?", "Low"),
    ("No rush, but I was wondering if you have a dark mode option.", "Low"),
    ("Just curious what payment methods you accept.", "Low"),
    ("I'd like to know more about your loyalty program sometime.", "Low"),
    ("Someday I'd like to upgrade my plan, no timeline in mind.", "Low"),
    ("Wondering if you ship internationally.", "Low"),
    ("The app is a bit slow to load sometimes, nothing major.", "Medium"),
    ("I noticed a small mistake on my invoice, can you take a look when you can?", "Medium"),
    ("My preferences keep resetting every time I log back in.", "Medium"),
    ("A feature I use often stopped showing up in the menu.", "Medium"),
    ("The colors on my order don't quite match what I saw online.", "Medium"),
    ("I'd appreciate a callback about a billing question this week.", "Medium"),
]

urg_test_df = pd.DataFrame(urgency_robust, columns=["combined_text", "human_label"])
urg_test_df["labels"] = urg_test_df["human_label"].map(urg2id)

urg_test_dataset = Dataset.from_pandas(urg_test_df[["combined_text", "labels"]])
urg_test_dataset = urg_test_dataset.map(tokenize_function, batched=True)
urg_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

urg_robust_out = trainer_urg.predict(urg_test_dataset)
urg_robust_preds = np.argmax(urg_robust_out.predictions, axis=1)

print(classification_report(urg_test_df["labels"], urg_robust_preds, target_names=urgency_labels, zero_division=0))

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

        High       0.00      0.00      0.00         6
         Low       0.33      1.00      0.50         6
      Medium       0.00      0.00      0.00         6

    accuracy                           0.33        18
   macro avg       0.11      0.33      0.17        18
weighted avg       0.11      0.33      0.17        18



In [ ]:
urgency_augment = [
    # High - real stakes, no trigger words
    ("Our entire team has been unable to log into the platform since 9am and we have a deadline today.", "High"),
    ("I just noticed two large payments leaving my account that I never approved.", "High"),
    ("The system went dark in the middle of processing customer orders.", "High"),
    ("My child's medical records disappeared from the portal right before an appointment.", "High"),
    ("We're a hospital and the scheduling tool stopped responding entirely.", "High"),
    ("Every user in my company got signed out at once and nobody can get back in.", "High"),
    ("I think my account was taken over, the email and phone number have both changed.", "High"),
    ("The payment gateway is rejecting every single transaction on our store right now.", "High"),
    ("A large sum was withdrawn and I have no record of authorizing it.", "High"),
    ("Production has stopped completely and we're losing money every minute this continues.", "High"),
    ("I need this resolved before end of day or we lose the client entirely.", "High"),
    ("The building's access system is offline and staff can't get inside.", "High"),
    # Medium - a real problem, not an emergency
    ("The dashboard loads a bit slower than it used to since last week.", "Medium"),
    ("A couple of numbers on my report look off compared to last month.", "Medium"),
    ("The mobile app sometimes logs me out for no clear reason.", "Medium"),
    ("Shipping took longer than the estimate I was given.", "Medium"),
    ("A setting I changed last week reverted back on its own.", "Medium"),
    ("I noticed the totals on two invoices don't quite line up.", "Medium"),
    ("The search feature sometimes misses results that should show up.", "Medium"),
    ("My subscription renewed a day earlier than I expected.", "Medium"),
    ("One of the buttons on the settings page doesn't respond the first time I click it.", "Medium"),
    ("The color scheme looks different on my phone than on my laptop.", "Medium"),
    ("A recent order arrived with the wrong size, could this be swapped?", "Medium"),
    ("I'd like someone to check why my usage numbers seem a little high this month.", "Medium"),
    # Low - genuinely casual, no pressure
    ("Just wanted to say the new update looks nice.", "Low"),
    ("Out of curiosity, do you have plans for a desktop version?", "Low"),
    ("No particular reason, just wondering what plans you offer.", "Low"),
    ("I'll probably switch plans eventually, just exploring options for now.", "Low"),
    ("Whenever it's convenient, I'd love a walkthrough of the reporting tools.", "Low"),
    ("Just checking if you have a referral program.", "Low"),
    ("Wondering aloud if there's a way to change the app icon.", "Low"),
    ("Not urgent at all, but do you support multiple languages?", "Low"),
    ("I might be interested in the enterprise plan down the line.", "Low"),
    ("Just browsing your FAQ, thought I'd ask if this is covered anywhere.", "Low"),
    ("No timeline in mind, but could you add a feature request for dark mode?", "Low"),
]

aug_df = pd.DataFrame(urgency_augment, columns=["combined_text", "human_label"])
aug_df["urgency_id"] = aug_df["human_label"].map(urg2id)

train_df_aug = pd.concat([train_df, aug_df[["combined_text", "urgency_id"]]], ignore_index=True)
print(train_df.shape, "->", train_df_aug.shape)

(6775, 24) -> (6810, 24)


In [ ]:
train_dataset_urg2 = Dataset.from_pandas(train_df_aug[["combined_text", "urgency_id"]])
train_dataset_urg2 = train_dataset_urg2.map(tokenize_function, batched=True)
train_dataset_urg2 = train_dataset_urg2.rename_column("urgency_id", "labels")
train_dataset_urg2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

urgency_model2 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

trainer_urg2 = Trainer(
    model=urgency_model2,
    args=training_args_urg,
    train_dataset=train_dataset_urg2,
    eval_dataset=test_dataset_urg,
    compute_metrics=compute_metrics
)

trainer_urg2.train()

Map:   0%|          | 0/6810 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.153337,0.126792,0.973436,0.972538,0.973450,0.973436
2,0.097957,0.059276,0.989965,0.989730,0.990050,0.989965
3,0.050553,0.055271,0.991145,0.991021,0.991056,0.991145


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1278, training_loss=0.1194242496445705, metrics={'train_runtime': 266.8315, 'train_samples_per_second': 76.565, 'train_steps_per_second': 4.79, 'total_flos': 676589304430080.0, 'train_loss': 0.1194242496445705, 'epoch': 3.0})

In [ ]:
urg_robust_out2 = trainer_urg2.predict(urg_test_dataset)
urg_robust_preds2 = np.argmax(urg_robust_out2.predictions, axis=1)
print(classification_report(urg_test_df["labels"], urg_robust_preds2, target_names=urgency_labels, zero_division=0))

              precision    recall  f1-score   support

        High       0.86      1.00      0.92         6
         Low       1.00      1.00      1.00         6
      Medium       1.00      0.83      0.91         6

    accuracy                           0.94        18
   macro avg       0.95      0.94      0.94        18
weighted avg       0.95      0.94      0.94        18



In [ ]:
trainer.save_model("./department_model")
tokenizer.save_pretrained("./department_model")

trainer_urg2.save_model("./urgency_model")
tokenizer.save_pretrained("./urgency_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./urgency_model/tokenizer_config.json', './urgency_model/tokenizer.json')

In [ ]:
from transformers import pipeline

department_pipe = pipeline("text-classification", model="./department_model", tokenizer="./department_model")
urgency_pipe = pipeline("text-classification", model="./urgency_model", tokenizer="./urgency_model")

def predict_ticket(text):
    dept_result = department_pipe(text)[0]
    urg_result = urgency_pipe(text)[0]

    dept_id = int(dept_result["label"].split("_")[-1])
    urg_id = int(urg_result["label"].split("_")[-1])

    return {
        "department": id2dept_v2[dept_id],
        "department_confidence": round(dept_result["score"], 3),
        "urgency": id2urg[urg_id],
        "urgency_confidence": round(urg_result["score"], 3)
    }

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
print(predict_ticket("I was billed twice for the same subscription this month."))
print(predict_ticket("My laptop screen keeps flickering and won't stay on."))
print(predict_ticket("Just wondering what your return policy looks like."))
print(predict_ticket("Our entire team got locked out right before a client call in ten minutes."))

{'department': 'Billing', 'department_confidence': 0.999, 'urgency': 'High', 'urgency_confidence': 0.978}
{'department': 'General', 'department_confidence': 0.598, 'urgency': 'High', 'urgency_confidence': 0.898}
{'department': 'General', 'department_confidence': 0.996, 'urgency': 'Low', 'urgency_confidence': 0.996}
{'department': 'General', 'department_confidence': 0.996, 'urgency': 'High', 'urgency_confidence': 0.999}


In [ ]:
dept_augment = [
    # Billing - money-related, no billing keywords
    ("Money got taken out of my account twice for one order.", "Billing"),
    ("I returned my item weeks ago and still haven't gotten my money back.", "Billing"),
    ("The amount on my statement doesn't match what I was quoted.", "Billing"),
    ("I was promised a discount that never showed up on my total.", "Billing"),
    ("Why did my total go up without any explanation this month?", "Billing"),
    ("I paid for one plan but got moved to a more expensive one.", "Billing"),
    ("Can someone explain the extra amount added to my last order?", "Billing"),
    ("I never received a confirmation that my money was returned.", "Billing"),
    ("The price I paid doesn't match the price shown on the site.", "Billing"),
    ("I was promised a free month but still got charged in full.", "Billing"),
    # Technical - broken things, no technical keywords
    ("My laptop screen goes black after a few minutes of use.", "Technical"),
    ("The app freezes every time I open the settings page.", "Technical"),
    ("Wi-Fi keeps dropping whenever I try to stream anything.", "Technical"),
    ("My phone gets extremely hot and loses battery fast.", "Technical"),
    ("The printer won't connect to my laptop anymore.", "Technical"),
    ("Sound cuts out randomly during video calls.", "Technical"),
    ("The camera shows a blank screen when I open the program.", "Technical"),
    ("My smart TV won't turn on and the light keeps blinking.", "Technical"),
    ("The update froze halfway and now the machine won't start.", "Technical"),
    ("Files take forever to load and everything feels extremely slow.", "Technical"),
    # Account - access/management, no account keywords
    ("I can't get into my profile and the reset email never arrives.", "Account"),
    ("Please close my membership at the end of this month.", "Account"),
    ("I want to change the email address linked to my profile.", "Account"),
    ("Someone else may have gotten into my profile recently.", "Account"),
    ("How do I delete my profile and all my data permanently?", "Account"),
    ("I forgot the phrase I use to access my profile.", "Account"),
    ("Please stop the automatic renewal on my plan.", "Account"),
    ("I want to upgrade my membership to the premium tier.", "Account"),
    ("My profile name is spelled wrong and I can't edit it.", "Account"),
    ("I'd like to merge my two profiles into one.", "Account"),
    # General - pre-sale/informational, no keywords from any list
    ("Do you ship internationally and how long does it take?", "General"),
    ("What are your store hours on weekends?", "General"),
    ("Is this model compatible with the older version of your product?", "General"),
    ("Which colors does the new version come in?", "General"),
    ("Do you have a student discount program?", "General"),
    ("How long does the warranty last on this item?", "General"),
    ("What's the difference between your two premium models?", "General"),
    ("Where's the nearest store where I could see this in person?", "General"),
    ("Are there any sales coming up this month?", "General"),
    ("Is there a manual available in Spanish?", "General"),
]

dept_aug_df = pd.DataFrame(dept_augment, columns=["combined_text", "human_label"])
dept_aug_df["department_id_v2"] = dept_aug_df["human_label"].map(dept2id_v2)

leaks = dept_aug_df["combined_text"].apply(find_leaks)
print("Leaked keywords found:", leaks.apply(len).sum())

train_df_dept_aug = pd.concat([train_df, dept_aug_df[["combined_text", "department_id_v2"]]], ignore_index=True)
print(train_df.shape, "->", train_df_dept_aug.shape)

Leaked keywords found: 2
(6775, 24) -> (6815, 24)


In [ ]:
leaked_rows = dept_aug_df[dept_aug_df["combined_text"].apply(lambda x: len(find_leaks(x)) > 0)]
print(leaked_rows[["combined_text", "human_label"]].to_string())
for text in leaked_rows["combined_text"]:
    print(text, "->", find_leaks(text))

                                                combined_text human_label
9  I was promised a free month but still got charged in full.     Billing
I was promised a free month but still got charged in full. -> ['charge', 'charged']


In [ ]:
dept_augment[9] = ("I was promised a free month but still had to pay the full amount.", "Billing")

dept_aug_df = pd.DataFrame(dept_augment, columns=["combined_text", "human_label"])
dept_aug_df["department_id_v2"] = dept_aug_df["human_label"].map(dept2id_v2)

leaks = dept_aug_df["combined_text"].apply(find_leaks)
print("Leaked keywords found:", leaks.apply(len).sum())

train_df_dept_aug = pd.concat([train_df, dept_aug_df[["combined_text", "department_id_v2"]]], ignore_index=True)
print(train_df.shape, "->", train_df_dept_aug.shape)

Leaked keywords found: 0
(6775, 24) -> (6815, 24)


In [ ]:
train_dataset_dept2 = Dataset.from_pandas(train_df_dept_aug[["combined_text", "department_id_v2"]])
train_dataset_dept2 = train_dataset_dept2.map(tokenize_function, batched=True)
train_dataset_dept2 = train_dataset_dept2.rename_column("department_id_v2", "labels")
train_dataset_dept2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model2 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=train_dataset_dept2,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer2.train()

Map:   0%|          | 0/6815 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.053475,0.043577,0.993506,0.993451,0.993600,0.993506
2,0.024507,0.023413,0.996458,0.996459,0.996460,0.996458
3,0.018679,0.020235,0.996458,0.996459,0.996460,0.996458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1278, training_loss=0.07044582241338697, metrics={'train_runtime': 267.7177, 'train_samples_per_second': 76.368, 'train_steps_per_second': 4.774, 'total_flos': 677098140687360.0, 'train_loss': 0.07044582241338697, 'epoch': 3.0})

In [ ]:
robust_dataset2 = Dataset.from_pandas(robust_df[["combined_text", "labels"]])
robust_dataset2 = robust_dataset2.map(tokenize_function, batched=True)
robust_dataset2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

dept_robust_out2 = trainer2.predict(robust_dataset2)
dept_robust_preds2 = np.argmax(dept_robust_out2.predictions, axis=1)

print(classification_report(robust_df["labels"], dept_robust_preds2, target_names=department_labels_v2, zero_division=0))

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

     Account       1.00      1.00      1.00        12
     Billing       0.92      1.00      0.96        12
     General       1.00      0.92      0.96        12
   Technical       1.00      1.00      1.00        12

    accuracy                           0.98        48
   macro avg       0.98      0.98      0.98        48
weighted avg       0.98      0.98      0.98        48



In [ ]:
trainer2.save_model("./department_model")
tokenizer.save_pretrained("./department_model")

trainer_urg2.save_model("./urgency_model")
tokenizer.save_pretrained("./urgency_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./urgency_model/tokenizer_config.json', './urgency_model/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

shutil.copytree("./department_model", "/content/drive/MyDrive/department_model", dirs_exist_ok=True)
shutil.copytree("./urgency_model", "/content/drive/MyDrive/urgency_model", dirs_exist_ok=True)

'/content/drive/MyDrive/urgency_model'

In [ ]:
!pip install -q gradio

import gradio as gr

def gradio_predict(text):
    result = predict_ticket(text)
    return (
        f"Department: {result['department']} (confidence: {result['department_confidence']})\n"
        f"Urgency: {result['urgency']} (confidence: {result['urgency_confidence']})"
    )

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=4, placeholder="Paste a customer support ticket here..."),
    outputs=gr.Textbox(label="Prediction"),
    title="SupportSense — Ticket Classifier & Router",
    description="Enter a customer support ticket to predict its department and urgency."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b8b9a4238ea71527d6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers gradio

from transformers import pipeline

department_pipe = pipeline("text-classification", model="/content/drive/MyDrive/department_model", tokenizer="/content/drive/MyDrive/department_model")
urgency_pipe = pipeline("text-classification", model="/content/drive/MyDrive/urgency_model", tokenizer="/content/drive/MyDrive/urgency_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

In [ ]:
department_labels_v2 = ['Account', 'Billing', 'General', 'Technical']
id2dept_v2 = {idx: label for idx, label in enumerate(department_labels_v2)}

urgency_labels = ['High', 'Low', 'Medium']
id2urg = {idx: label for idx, label in enumerate(urgency_labels)}

In [ ]:
def predict_ticket(text):
    dept_result = department_pipe(text)[0]
    urg_result = urgency_pipe(text)[0]

    dept_id = int(dept_result["label"].split("_")[-1])
    urg_id = int(urg_result["label"].split("_")[-1])

    return {
        "department": id2dept_v2[dept_id],
        "department_confidence": round(dept_result["score"], 3),
        "urgency": id2urg[urg_id],
        "urgency_confidence": round(urg_result["score"], 3)
    }

print(predict_ticket("My laptop screen keeps flickering and won't stay on."))

{'department': 'Technical', 'department_confidence': 1.0, 'urgency': 'High', 'urgency_confidence': 0.898}


In [ ]:
import gradio as gr

def gradio_predict(text):
    result = predict_ticket(text)
    return (
        f"Department: {result['department']} (confidence: {result['department_confidence']})\n"
        f"Urgency: {result['urgency']} (confidence: {result['urgency_confidence']})"
    )

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=4, placeholder="Paste a customer support ticket here..."),
    outputs=gr.Textbox(label="Prediction"),
    title="SupportSense — Ticket Classifier & Router",
    description="Enter a customer support ticket to predict its department and urgency."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ceab23352f94e1f4d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
